In [5]:
import re
import numpy as np
import pandas as pd

input_file = "/share/home/zhoujb/project/riceDataComb/processData/dataFromCowpea/Phenotypes for GWAS-upload public database.xlsx"
output_file = "/share/home/zhoujb/project/riceDataComb/processData/dataFromCowpea/phen_matrix_2.txt"

In [10]:
df = pd.read_excel(input_file)

# 删除无意义 ID 列
df = df.drop(columns=["ID"], errors="ignore")

# 如果担心有些 "-" 没被识别，再保险替换一次
df = df.replace("-", np.nan)

def parse_column(col):
    col = str(col).strip()

    # 只保留 HZ 或 GZ 开头的列，支持 HZ xxx / HZ-xxx / HZ_xxx
    m = re.match(r"^(HZ|GZ)[\s_-]+(.+)$", col, flags=re.I)
    if not m:
        return None

    region = m.group(1).upper()
    trait = m.group(2).strip()

    # 去除末尾单位，例如 (mg/g)、(g/kg)
    trait = re.sub(r"\s*\([^)]*\)\s*$", "", trait).strip()

    # 清理多余空格
    trait = re.sub(r"\s+", " ", trait)

    # 修正常见大小写和文件中的明显拼写不一致
    trait = re.sub(r"\bStrach\b", "Starch", trait, flags=re.I)
    trait = re.sub(r"\bprotein\b", "Protein", trait, flags=re.I)
    trait = re.sub(r"\bcontent\b", "content", trait, flags=re.I)

    # 文件中 HZ 为 Pod Protein content，GZ 为 Pod protein，统一名称
    trait = re.sub(r"^Pod Protein$", "Pod Protein content", trait, flags=re.I)

    trait_key = trait.lower()

    return region, trait_key, trait


# 提取 HZ/GZ 表型列信息
records = []

for col in df.columns:
    parsed = parse_column(col)
    if parsed is not None:
        region, trait_key, trait = parsed
        records.append({
            "old_col": col,
            "region": region,
            "trait_key": trait_key,
            "trait": trait
        })

info = pd.DataFrame(records)

# 找到 HZ 和 GZ 都出现的表型
common_traits = (
    info.groupby("trait_key")["region"]
    .nunique()
    .loc[lambda x: x == 2]
    .index
)

# 优先使用 HZ 中的表型名称作为最终列名
trait_names = (
    info[info["trait_key"].isin(common_traits)]
    .assign(region_order=lambda x: x["region"].map({"HZ": 0, "GZ": 1}))
    .sort_values(["trait_key", "region_order"])
    .drop_duplicates("trait_key")
    .set_index("trait_key")["trait"]
    .to_dict()
)

# 构建 long 表
long_list = []

for region in ["HZ", "GZ"]:
    sub = info[
        (info["region"] == region) &
        (info["trait_key"].isin(common_traits))
    ].drop_duplicates("trait_key")

    rename_dict = {
        row["old_col"]: trait_names[row["trait_key"]]
        for _, row in sub.iterrows()
    }

    tmp = df[["Accessions"] + sub["old_col"].tolist()].rename(columns=rename_dict)
    tmp.insert(1, "region", region)

    long_list.append(tmp)

long_df = pd.concat(long_list, ignore_index=True)

# 保存结果
long_df.to_csv(output_file, sep="\t", index=False)

print("完成！")
print("保留的共同表型：")
print(long_df.columns.tolist())
print(f"输出文件：{output_file}")

完成！
保留的共同表型：
['Accessions', 'region', 'Pod length', 'Pod Sugar content', 'Pod Starch content', 'Pod Protein content', 'Seed Sugar', 'Seed Starch', 'Seed Protein', 'TSW']
输出文件：/share/home/zhoujb/project/riceDataComb/processData/dataFromCowpea/phen_matrix_2.txt


/tmp/ipykernel_237001/1185621417.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace("-", np.nan)
